In [2]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator, PercentFormatter
import seaborn as sns
from IPython.display import display

In [3]:
TASK = Path.cwd().parent
ROOT = TASK.parent
INPUT, OUTPUT = TASK/'input', TASK/'output'
CONFIG = json.loads((ROOT/'config.json').read_text())
Y, SEED = CONFIG['last_year'], CONFIG['seed']
REFERENCES = ['age_0','lag_3','lag_5']
LABELS = {'age_0':r'$\rho_S(R_0,R_t)$', 'lag_3':r'$\rho_S(R_{t-3},R_t)$', 'lag_5':r'$\rho_S(R_{t-5},R_t)$'}
COLORS = dict(zip(REFERENCES,sns.color_palette('colorblind',3)))
LINES = dict(zip(REFERENCES,['-','--','-.']))
MARKERS = dict(zip(REFERENCES,['o','s','^']))
sns.set_theme(style='ticks',context='paper',font_scale=1.15,
              rc={'pdf.fonttype':42,'ps.fonttype':42,'savefig.dpi':300})

In [4]:
# annual ranks of the fixed complete-career DBLP cohort; zeros retain average ranks
ranks = pd.read_csv(INPUT/'rank_mixing.csv')
ranks = ranks.loc[ranks.source.eq('empirical_complete') & ranks.measure.eq('annual') & ranks.reference.isin(REFERENCES)].copy()
ranks = ranks.sort_values(['reference','age'])
ranks.to_csv(OUTPUT/'dblp_rank_mixing.csv',index=False)

In [ ]:
# pooled empirical follow-up for both publication transitions
MIN_REENTRY_RISK = 20
dropout = pd.read_csv(INPUT/'dropout_by_productivity.csv')
dropout = dropout.loc[dropout.source.eq('empirical_available') & dropout.stage.eq('pooled')].copy()
dropout['decile'] = dropout['bin'] + 1
reentry = pd.read_csv(INPUT/'restart_hazard.csv')
reentry = reentry.loc[reentry.source.eq('empirical_available') & reentry.n_at_risk.ge(MIN_REENTRY_RISK)]

In [5]:
summary = ranks.groupby('reference',sort=False).agg(start_age=('age','min'),end_age=('age','max'), n=('n','first')).reindex(REFERENCES)
summary['terminal_rho'] = ranks.loc[ranks.age.eq(Y)].set_index('reference').rho
summary['measure'] = 'annual'
summary['cohort'] = 'DBLP complete careers'
summary.to_csv(OUTPUT/'dblp_rank_mixing_summary.csv')
display(summary)

,start_age,end_age,n,terminal_rho,measure,cohort
reference,,,,,,
age_0,0,20,510,0.187294,annual,DBLP complete careers
lag_3,3,20,510,0.653156,annual,DBLP complete careers
lag_5,5,20,510,0.582696,annual,DBLP complete careers


In [ ]:
def plot_rank(ax, legend_size=None):
    for reference in REFERENCES:
        data = ranks.loc[ranks.reference.eq(reference)].sort_values('age')
        sns.lineplot(data=data, x='age', y='rho', ax=ax, color=COLORS[reference],
                     linestyle=LINES[reference], marker=MARKERS[reference],
                     markersize=3.5, markevery=2, linewidth=1.4,
                     estimator=None, errorbar=None, label=LABELS[reference])

    ax.set(xlabel='Career year, $t$', ylabel=r'Spearman $\rho$',
           xlim=(0, Y), ylim=(0, 1.05))
    ax.xaxis.set_major_locator(MaxNLocator(integer=True, nbins=5))
    ax.set_yticks([0, .25, .5, .75, 1])
    ax.legend(frameon=False, loc='upper right', handlelength=2.2,
              labelspacing=.25, borderaxespad=.3, fontsize=legend_size)


def plot_transition(ax, data, x, y, xlabel, xticks):
    sns.lineplot(data=data, x=x, y=y, ax=ax, estimator=None, errorbar=None,
                 color='.2', marker='o', markersize=3, linewidth=1.15)
    ax.set(xlabel=xlabel, ylabel='', xticks=xticks, ylim=(0,None))
    ax.yaxis.set_major_locator(MaxNLocator(3, steps=[1,2,2.5,5,10]))
    ax.yaxis.set_major_formatter(PercentFormatter(1, decimals=0))

In [ ]:
fig, ax = plt.subplots(figsize=(5.6, 2.5))
plot_rank(ax)
sns.despine(ax=ax)
fig.tight_layout(pad=.35)
for extension in ['pdf', 'png']:
    fig.savefig(OUTPUT/f'dblp_rank_mixing.{extension}', bbox_inches='tight',
                pad_inches=.02, facecolor='white')
plt.show()
plt.close(fig)

In [ ]:
for layout, size in [('wide',(7.1,2.05)), ('column',(3.42,3.45))]:
    with plt.rc_context({'axes.titlesize':8.5,'axes.labelsize':8,
                         'xtick.labelsize':7,'ytick.labelsize':7,
                         'xtick.major.pad':2,'ytick.major.pad':2,
                         'axes.labelpad':3,'axes.titlepad':5}):
        fig = plt.figure(figsize=size, layout='constrained')
        if layout == 'wide':
            grid = fig.add_gridspec(1,3,width_ratios=[2.2,1,1])
            rank_ax, dropout_ax, reentry_ax = (fig.add_subplot(spec) for spec in (grid[0,0],grid[0,1],grid[0,2]))
        else:
            grid = fig.add_gridspec(2,2,height_ratios=[1.65,1])
            rank_ax = fig.add_subplot(grid[0,:])
            dropout_ax, reentry_ax = (fig.add_subplot(spec) for spec in (grid[1,0],grid[1,1]))
        plot_rank(rank_ax, legend_size=7)
        plot_transition(dropout_ax, dropout, 'decile', 'probability', 'Output decile',
                        [1,5,10])
        plot_transition(reentry_ax, reentry, 'duration', 'restart_probability',
                        'Consecutive zero years', reentry.duration)
        for ax, label in zip((rank_ax,dropout_ax,reentry_ax),
                             ('A  Rank persistence','B  Dropout','C  Reentry')):
            ax.set_title(label, loc='left', fontweight='bold')
        sns.despine(fig=fig)
        for extension in ['pdf','png']:
            fig.savefig(OUTPUT/f'pnas_dblp_temporal_{layout}.{extension}', facecolor='white')
        plt.show()
        plt.close(fig)